# Module 0: Environment Setup

## Learning Objectives
- Create a Medallion Architecture database with DQ schema
- Understand the 4 source systems and their data quality characteristics
- Load raw data exactly as it arrives from each source (no transformation)
- Identify intentional DQ issues seeded for later modules

## Business Scenario

You are the **Data Quality Lead** at a Saudi Arabian holding company. The company operates across multiple subsidiaries and must consolidate customer and transaction data from **4 different source systems** into a single Data Warehouse.

Each source has different schemas, different quality levels, and different business ownership. Your job: build a DQ monitoring framework that catches issues at every layer.

---

> **Role Required:** `ACCOUNTADMIN` (this module only). All subsequent modules use `CORP_DQ_ADMIN`.

> **Time:** ~20 minutes

---
## Step 1: Create Database and Schemas

In [ ]:
CREATE DATABASE IF NOT EXISTS CORP_DWH
    COMMENT = 'Corporate Data Warehouse - Snowflake DQ Monitoring HOL';

CREATE SCHEMA IF NOT EXISTS CORP_DWH.RAW COMMENT = 'Bronze: raw data as-is from source systems';
CREATE SCHEMA IF NOT EXISTS CORP_DWH.SILVER COMMENT = 'Silver: cleansed, unified, scored (Dynamic Tables)';
CREATE SCHEMA IF NOT EXISTS CORP_DWH.GOLD COMMENT = 'Gold: business-ready dimensions and facts (dbt models)';
CREATE SCHEMA IF NOT EXISTS CORP_DWH.DQ COMMENT = 'Data Quality: DMFs, rules catalog, procedures, alerts';

---
## Step 2: Create Roles

In [ ]:
CREATE ROLE IF NOT EXISTS CORP_DQ_ADMIN;
GRANT USAGE ON DATABASE CORP_DWH TO ROLE CORP_DQ_ADMIN;
GRANT USAGE ON ALL SCHEMAS IN DATABASE CORP_DWH TO ROLE CORP_DQ_ADMIN;
GRANT ALL PRIVILEGES ON SCHEMA CORP_DWH.RAW TO ROLE CORP_DQ_ADMIN;
GRANT ALL PRIVILEGES ON SCHEMA CORP_DWH.SILVER TO ROLE CORP_DQ_ADMIN;
GRANT ALL PRIVILEGES ON SCHEMA CORP_DWH.GOLD TO ROLE CORP_DQ_ADMIN;
GRANT ALL PRIVILEGES ON SCHEMA CORP_DWH.DQ TO ROLE CORP_DQ_ADMIN;
GRANT CREATE DATA METRIC FUNCTION ON SCHEMA CORP_DWH.DQ TO ROLE CORP_DQ_ADMIN;
GRANT CREATE DYNAMIC TABLE ON SCHEMA CORP_DWH.SILVER TO ROLE CORP_DQ_ADMIN;
GRANT CREATE DYNAMIC TABLE ON SCHEMA CORP_DWH.GOLD TO ROLE CORP_DQ_ADMIN;
GRANT USAGE ON WAREHOUSE COMPUTE_WH TO ROLE CORP_DQ_ADMIN;
GRANT APPLICATION ROLE SNOWFLAKE.DATA_QUALITY_MONITORING_VIEWER TO ROLE CORP_DQ_ADMIN;
GRANT EXECUTE DATA METRIC FUNCTION ON ACCOUNT TO ROLE CORP_DQ_ADMIN;
GRANT DATABASE ROLE SNOWFLAKE.DATA_METRIC_USER TO ROLE CORP_DQ_ADMIN;
GRANT ROLE CORP_DQ_ADMIN TO ROLE ACCOUNTADMIN;

> **What this does:** Creates the CORP_DQ_STEWARD role with read-only access to data + write access to the DQ schema. This role is for data stewards who manage rules but don't administer the pipeline.

In [ ]:
CREATE ROLE IF NOT EXISTS CORP_DQ_STEWARD;
GRANT USAGE ON DATABASE CORP_DWH TO ROLE CORP_DQ_STEWARD;
GRANT USAGE ON ALL SCHEMAS IN DATABASE CORP_DWH TO ROLE CORP_DQ_STEWARD;
GRANT ALL PRIVILEGES ON SCHEMA CORP_DWH.DQ TO ROLE CORP_DQ_STEWARD;
GRANT CREATE DATA METRIC FUNCTION ON SCHEMA CORP_DWH.DQ TO ROLE CORP_DQ_STEWARD;
GRANT SELECT ON ALL TABLES IN SCHEMA CORP_DWH.RAW TO ROLE CORP_DQ_STEWARD;
GRANT SELECT ON ALL TABLES IN SCHEMA CORP_DWH.SILVER TO ROLE CORP_DQ_STEWARD;
GRANT SELECT ON ALL TABLES IN SCHEMA CORP_DWH.GOLD TO ROLE CORP_DQ_STEWARD;
GRANT USAGE ON WAREHOUSE COMPUTE_WH TO ROLE CORP_DQ_STEWARD;
GRANT EXECUTE DATA METRIC FUNCTION ON ACCOUNT TO ROLE CORP_DQ_STEWARD;
GRANT APPLICATION ROLE SNOWFLAKE.DATA_QUALITY_MONITORING_VIEWER TO ROLE CORP_DQ_STEWARD;
GRANT ROLE CORP_DQ_STEWARD TO ROLE CORP_DQ_ADMIN;

---
## Step 3: Enable Cross-Region Inference + Notification Integration

> **ACTION REQUIRED:** Replace `<<YOUR_EMAIL>>` below with your actual Snowflake-verified email address before running this cell. The email must match a verified address in your Snowflake account.

In [ ]:
ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'ANY_REGION';

CREATE OR REPLACE NOTIFICATION INTEGRATION CORP_DQ_ALERTS
    TYPE = EMAIL ENABLED = TRUE
    ALLOWED_RECIPIENTS = ('<<YOUR_EMAIL>>');
GRANT USAGE ON INTEGRATION CORP_DQ_ALERTS TO ROLE CORP_DQ_ADMIN;

---
## Step 4: Switch to Lab Role

In [ ]:
USE ROLE CORP_DQ_ADMIN;
USE DATABASE CORP_DWH;
USE WAREHOUSE COMPUTE_WH;

---
## Source System 1: ERP (SAP S/4HANA)

| Attribute | Detail |
|-----------|--------|
| **System** | SAP S/4HANA -- Financial master data |
| **Feed method** | Nightly batch CSV export via SFTP |
| **Data owner** | Finance Department |
| **Quality level** | HIGH -- structured, validated at input |
| **Refresh frequency** | Daily (overnight batch) |

**Business context:** ERP is the "system of record" for billing. If a customer exists in ERP, they receive invoices. IBAN is mandatory because payments are initiated from this system. This is your most trusted source.

**Known issues (minor):**
- One record has leading/trailing spaces in NATIONAL_ID (copy-paste artifact)
- One record is missing the English name (Arabic-only entry)
- Phone format varies: some have +966, others have local 05 prefix

In [ ]:
CREATE OR REPLACE TABLE CORP_DWH.RAW.STG_CUSTOMERS_ERP (
    RAW_ID NUMBER AUTOINCREMENT,
    CUSTOMER_NAME_AR STRING COMMENT 'Arabic name from SAP',
    CUSTOMER_NAME_EN STRING COMMENT 'English name (optional in SAP)',
    NATIONAL_ID STRING COMMENT 'Saudi ID - should be 10 digits',
    IBAN STRING COMMENT 'Bank account for payments',
    EMAIL STRING,
    PHONE STRING COMMENT 'Format varies: +966, 05, 00966',
    CITY_CODE STRING COMMENT 'SAP city code (RUH, JED, DMM, MKH)',
    CREATED_IN_SOURCE STRING COMMENT 'Original creation date in SAP',
    LOADED_AT TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP(),
    SOURCE_FILE STRING
);

INSERT INTO CORP_DWH.RAW.STG_CUSTOMERS_ERP
    (CUSTOMER_NAME_AR, CUSTOMER_NAME_EN, NATIONAL_ID, IBAN, EMAIL, PHONE, CITY_CODE, CREATED_IN_SOURCE, SOURCE_FILE)
VALUES
    ('Abdullah AR', 'Abdullah Al-Rashid', '1087654321', 'SA0380000000608010167519', 'a.rashid@acme.sa', '+966501234567', 'RUH', '2024-01-15', 'ERP_BATCH_001.csv'),
    ('Fatima AR', 'Fatima Al-Zahrani', '1098765432', 'SA4420000001234567891234', 'f.zahrani@acme.sa', '+966512345678', 'JED', '2024-02-20', 'ERP_BATCH_001.csv'),
    ('Mohammed AR', 'Mohammed Al-Otaibi', ' 1076543210 ', 'SA6680000000608010167520', 'mo.otaibi@acme.sa', '0523456789', 'DMM', '2024-03-10', 'ERP_BATCH_002.csv'),
    ('Nora AR', NULL, '2087654321', 'SA7780000000608010167521', 'n.shamari@acme.sa', '00966534567890', 'RUH', '15/04/2024', 'ERP_BATCH_002.csv'),
    ('Khalid AR', 'Khalid Al-Harbi', '1065432109', 'SA8880000000608010167522', 'k.harbi@acme.sa', '+966545678901', 'MKH', '2024-05-01', 'ERP_BATCH_003.csv'),
    ('Reem AR', 'Reem Al-Tamimi', '1043210987', 'SA5580000000608010167528', 'r.tamimi@acme.sa', '+966534567891', 'RUH', '2024-01-01', 'ERP_BATCH_001.csv');

---
## Source System 2: CRM (Salesforce)

| Attribute | Detail |
|-----------|--------|
| **System** | Salesforce Sales Cloud |
| **Feed method** | API sync every 2 hours (JSON payloads) |
| **Data owner** | Sales Department |
| **Quality level** | LOW -- sales reps enter data quickly, skip optional fields |
| **Refresh frequency** | Every 2 hours (near-real-time) |

**Business context:** CRM captures leads, contacts, and opportunities. Many records are "soft" -- a salesperson adds a contact from a business card or phone call. National ID is not mandatory in Salesforce (it's not needed for sales activities). No IBAN because CRM doesn't handle payments.

**Known issues (significant):**
- 3 out of 6 records have NULL National IDs (field not mandatory in Salesforce)
- 2 records are duplicates of ERP customers (same person registered separately in CRM with personal email)
- City names are free-text (not standardized codes like ERP)
- One record has lowercase city name ('tabuk' instead of 'Tabuk')

In [ ]:
CREATE OR REPLACE TABLE CORP_DWH.RAW.STG_CUSTOMERS_CRM (
    RAW_ID NUMBER AUTOINCREMENT,
    FULL_NAME STRING COMMENT 'Single name field (no AR/EN split)',
    NATIONAL_ID STRING COMMENT 'Often NULL - not mandatory in SF',
    EMAIL STRING,
    MOBILE STRING COMMENT 'Local format (05xxxxxxxx)',
    CITY STRING COMMENT 'Free text - not standardized',
    REGISTRATION_DATE STRING COMMENT 'String date from API',
    LOADED_AT TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP(),
    SOURCE_FILE STRING
);

INSERT INTO CORP_DWH.RAW.STG_CUSTOMERS_CRM
    (FULL_NAME, NATIONAL_ID, EMAIL, MOBILE, CITY, REGISTRATION_DATE, SOURCE_FILE)
VALUES
    ('Sara Al-Dosari', NULL, 's.dosari@acme.sa', '0556789012', 'Riyadh', '2024-06-15', 'CRM_Q2.json'),
    ('Omar Al-Qahtani', NULL, 'o.qahtani@acme.sa', '0567890123', 'Jeddah', '2024-07-20', 'CRM_Q2.json'),
    ('Huda Al-Shehri', NULL, 'h.shehri@acme.sa', '0578901234', 'tabuk', '2024-08-10', 'CRM_Q3.json'),
    ('Abdullah Al-Rashid', '1087654321', 'abdullah.r@gmail.com', '0501234567', 'Riyadh', '2024-01-20', 'CRM_Q1.json'),
    ('Mohammed Al-Otaibi', '1076543210', 'mohammed.o@yahoo.com', '0523456789', 'Dammam', '2024-03-15', 'CRM_Q1.json'),
    ('Nouf Al-Qahtani', '2043210987', 'n.qahtani2@acme.sa', '578901235', 'Jeddah', '2025-01-01', 'CRM_Q4.json');

---
## Source System 3: Government Portal (GOSI / Absher)

| Attribute | Detail |
|-----------|--------|
| **System** | GOSI (Social Insurance) and Absher (Residency) portal exports |
| **Feed method** | Manual monthly Excel upload by Compliance team |
| **Data owner** | Compliance / Legal Department |
| **Quality level** | VERY LOW -- PDF-to-Excel conversion introduces OCR artifacts |
| **Refresh frequency** | Monthly (manual process) |

**Business context:** Saudi holding companies must cross-reference their records against government registries. GOSI provides social insurance data for Saudi employees; Absher provides residency data for expat workers. The compliance team downloads PDFs from government portals, converts them to Excel (often via OCR), and uploads the result. This process introduces severe formatting errors.

**Known issues (critical -- all records have invalid National IDs):**
- `98765` -- too short (5 digits instead of 10, OCR truncated leading digits)
- `30876543210` -- too long (11 digits, OCR merged two fields)
- `ABC1234567` -- contains letters (OCR misread Arabic digits as Latin characters)

In [ ]:
CREATE OR REPLACE TABLE CORP_DWH.RAW.STG_GOV_PORTAL (
    RAW_ID NUMBER AUTOINCREMENT,
    PERSON_NAME STRING COMMENT 'Name from government record',
    NATIONAL_ID STRING COMMENT 'Severely corrupted by OCR/PDF extraction',
    EMAIL STRING,
    PHONE STRING,
    GOV_SERVICE STRING COMMENT 'Which government service (GOSI, ABSHER)',
    EXTRACT_DATE STRING COMMENT 'Date of the government extract',
    LOADED_AT TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP(),
    SOURCE_FILE STRING
);

INSERT INTO CORP_DWH.RAW.STG_GOV_PORTAL
    (PERSON_NAME, NATIONAL_ID, EMAIL, PHONE, GOV_SERVICE, EXTRACT_DATE, SOURCE_FILE)
VALUES
    ('Tariq Al-Mutairi', '98765', 't.mutairi@acme.sa', '+966589012345', 'GOSI', '2024-09-01', 'GOV_EXTRACT_SEP.xlsx'),
    ('Layla Al-Ghamdi', '30876543210', 'l.ghamdi@acme.sa', '+966590123456', 'ABSHER', '2024-09-15', 'GOV_EXTRACT_SEP.xlsx'),
    ('Yousef Al-Dossary', 'ABC1234567', 'y.dossary@acme.sa', '+966501234568', 'GOSI', '2024-10-01', 'GOV_EXTRACT_OCT.xlsx');

---
## Source System 4: Bank Transaction Feed

| Attribute | Detail |
|-----------|--------|
| **System** | Core banking reconciliation file |
| **Feed method** | Daily SFTP drop (CSV) by Treasury |
| **Data owner** | Treasury Department |
| **Quality level** | MEDIUM -- structured but dates/amounts arrive as strings |
| **Refresh frequency** | Daily (morning drop, must be processed by noon) |

**Business context:** Bank feeds are the financial truth. Every SAR must reconcile with the general ledger. But the raw format from the bank requires parsing: dates come in multiple formats, amounts include commas and currency prefixes, and the bank uses single-character type codes.

**Known issues:**
- Row 5: BLANK customer reference (empty string, not NULL -- subtle!)
- Row 6: Negative amount (incorrectly coded refund)
- Row 4: Stale record loaded 3 days ago (SLA is 2 hours)
- Mixed date formats: ISO (2025-08-15) and DD/MM/YYYY (15/08/2025)
- Amounts as strings: '7,500.50' and 'SAR 22000'

In [ ]:
CREATE OR REPLACE TABLE CORP_DWH.RAW.STG_TRANSACTIONS (
    RAW_ID NUMBER AUTOINCREMENT,
    CUSTOMER_REF STRING COMMENT 'Customer reference from bank',
    TXN_DATE_STR STRING COMMENT 'Date as STRING - mixed formats!',
    AMOUNT_STR STRING COMMENT 'Amount as STRING - has commas, prefix!',
    CURRENCY STRING,
    TXN_TYPE_CODE STRING COMMENT 'Single char: P=Payment, I=Invoice, T=Transfer, R=Refund',
    LOADED_AT TIMESTAMP_LTZ COMMENT 'When we received the file',
    SOURCE_FILE STRING
);

INSERT INTO CORP_DWH.RAW.STG_TRANSACTIONS
    (CUSTOMER_REF, TXN_DATE_STR, AMOUNT_STR, CURRENCY, TXN_TYPE_CODE, LOADED_AT, SOURCE_FILE)
VALUES
    ('CUST-001', '2025-08-15', '15000.00', 'SAR', 'P', CURRENT_TIMESTAMP(), 'BANK_FEED.csv'),
    ('CUST-002', '15/08/2025', '7,500.50', 'SAR', 'I', CURRENT_TIMESTAMP(), 'BANK_FEED.csv'),
    ('CUST-003', '2025-08-14', 'SAR 22000', 'SAR', 'T', CURRENT_TIMESTAMP(), 'BANK_FEED.csv'),
    ('CUST-004', '2025-08-13', '3200.00', 'SAR', 'R', DATEADD(DAY, -3, CURRENT_TIMESTAMP()), 'BANK_FEED.csv'),
    ('', '2025-08-16', '8900.00', 'SAR', 'P', CURRENT_TIMESTAMP(), 'BANK_FEED.csv'),
    ('CUST-001', '2025-08-16', '-500.00', 'SAR', 'R', CURRENT_TIMESTAMP(), 'BANK_FEED.csv');

---
## Verification: Row Counts

In [ ]:
SELECT 'RAW.STG_CUSTOMERS_ERP' AS TABLE_NAME, COUNT(*) AS ROW_COUNT FROM CORP_DWH.RAW.STG_CUSTOMERS_ERP
UNION ALL SELECT 'RAW.STG_CUSTOMERS_CRM', COUNT(*) FROM CORP_DWH.RAW.STG_CUSTOMERS_CRM
UNION ALL SELECT 'RAW.STG_GOV_PORTAL', COUNT(*) FROM CORP_DWH.RAW.STG_GOV_PORTAL
UNION ALL SELECT 'RAW.STG_TRANSACTIONS', COUNT(*) FROM CORP_DWH.RAW.STG_TRANSACTIONS
ORDER BY TABLE_NAME;

---
## Setup Complete

You now have **RAW data only** -- exactly as it arrives from each source system. No transformations have been applied yet.

| Source | Table | Rows | Quality | Owner |
|--------|-------|------|---------|-------|
| SAP ERP | STG_CUSTOMERS_ERP | 6 | High | Finance |
| Salesforce CRM | STG_CUSTOMERS_CRM | 6 | Low | Sales |
| Gov Portal (GOSI/Absher) | STG_GOV_PORTAL | 3 | Very Low | Compliance |
| Bank Feed | STG_TRANSACTIONS | 6 | Medium | Treasury |

**Next:** Open `0B_DATA_PIPELINE` to build the transformation pipeline using Dynamic Tables and dbt.